# Requests

In [ ]:
import datetime
from time import sleep
import time
import csv
import re
import os

import requests
from selenium import webdriver
import chromedriver_autoinstaller
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
import undetected_chromedriver as uc

import project_secrets


In [ ]:
DATE_NOW = datetime.datetime.now().strftime('%y%m%d')
DATE_NOW

In [ ]:
url = 'https://app.musicleague.com/home/'

response = requests.get(url)
print(response.status_code)

response.text

# Selenium

In [ ]:
# !pip install undetected-chromedriver

In [ ]:
# !pip install setuptools

In [ ]:
OUTPUT_FOLDER = 'output'
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

In [ ]:
driver = uc.Chrome(version_main = 145)
driver.delete_all_cookies()

driver.get("https://app.musicleague.com/home/")
driver.maximize_window()
sleep(2)

driver.find_element(By.CSS_SELECTOR, ".btn.border-2.btn-outline-light.fw-semibold").click()

sleep(2)

wait = WebDriverWait(driver, 10)

search_box = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, ".e-91185-form-input.e-91185-baseline.e-91185-form-control.encore-text-body-medium.e-91185-focus-border"))
)
search_box.send_keys(project_secrets.email_auth)

search_button = driver.find_element(By.CSS_SELECTOR, ".e-91185-baseline.e-91185-overflow-wrap-anywhere.e-91185-button-primary__inner.encore-bright-accent-set.e-91185-button--medium")
search_button.click()

a = input('let me know when you are done')
if a.lower() == 'yes':
    print('continuing')

html = driver.page_source


element = driver.find_element(By.XPATH, "//a[text()='all bangers all the time ✨']")

driver.execute_script("arguments[0].scrollIntoView()", element)
driver.find_element(By.XPATH, "//a[text()='all bangers all the time ✨']").click()

sleep(5)

data = []


links = driver.find_elements(By.CSS_SELECTOR, ".stretched-link.text-body-tertiary.text-decoration-none")

urls = [link.get_attribute('href') for link in links]
urls = [i for i in urls if 'app.musicleague.com' in i]

counter = 1
for url in urls:

    driver.get(url)
    wait.until(
        EC.presence_of_element_located((
            # By.TAG_NAME, 'body'
            By.CSS_SELECTOR, ".card-footer.bg-body.collapse.show"
        ))
    )
    # Extract all visible text
    text = driver.find_element(By.TAG_NAME, 'body').text
    # store result
    data.append([url, text])
    
    # save html
    html = driver.page_source
    with open(f'{OUTPUT_FOLDER}/scraped_data/page_{counter}.html', 'w', encoding = 'utf-8') as f:
        f.write(html)
        counter += 1
        
    driver.back()
    wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".stretched-link.text-body-tertiary.text-decoration-none"))
    )
#    time.sleep(1)

with open(f'{OUTPUT_FOLDER}/scraped_data/{DATE_NOW}_scraped_pages.csv', 'w', newline = '', encoding = 'utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['url', 'text'])
    writer.writerows(data)
    writer.writerows('\n\n==========================\n\n')
       